# Interactive Semble Benchmark Runner
### Select Any Repository & Compare Semble vs. VQ-bench AST vs. VQ-bench CFG

This notebook allows you to select any repository from the official **Semble Benchmark Suite** (`MinishLab/semble`) and run a live head-to-head evaluation across three code-search paradigms:

1. **Semble Baseline**: Model2Vec `potion-code-16M` (Float32) + BM25 RRF on whole-function AST chunks.
2. **VQ-bench AST Solution**: Whole-function AST chunks + $1.35$ bits/dim dictionary quantization ($95.8\%$ RAM savings) + Two-stage hybrid search.
3. **VQ-bench CFG Solution**: Control Flow Graph (CFG) basic-block partitioning ($\ge 38$ chars) + DFG def-use chains + symbol-boosted anisotropic RRF at $1.35$ b/d.

In [ ]:
import os
import sys
import re
import time
import math
import json
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from model2vec import StaticModel

# Ensure project root is in path
sys.path.insert(0, os.path.abspath("."))
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../examples"))

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"[✓] Compute Device: {device}")

# Load Encoders
print("[*] Loading potion-code-16M (Model2Vec) & ColBERTv2...")
colbert_tok = AutoTokenizer.from_pretrained('colbert-ir/colbertv2.0')
colbert_mod = AutoModel.from_pretrained('colbert-ir/colbertv2.0').to(device).eval()
potion_code = StaticModel.from_pretrained("MinishLab/potion-code-16M")
print("[✓] Models initialized successfully!")

## 1. Select a Benchmark Repository

Below is the list of available repositories from Semble's benchmark suite. Change `SELECTED_REPO` to any repository you want to test!

In [ ]:
# Robust path resolution for repos and annotations
if os.path.exists("scratch/semble_benchmarks/repos.json"):
    repos_json_path = "scratch/semble_benchmarks/repos.json"
    annot_dir = "scratch/semble_benchmarks/annotations"
    repos_dir = "scratch/semble_repos"
else:
    repos_json_path = "../scratch/semble_benchmarks/repos.json"
    annot_dir = "../scratch/semble_benchmarks/annotations"
    repos_dir = "../scratch/semble_repos"

with open(repos_json_path, "r") as f:
    all_repos = json.load(f)

available_repos = []
for r in all_repos:
    rname = r["name"]
    if os.path.exists(os.path.join(annot_dir, f"{rname}.json")) and os.path.exists(os.path.join(repos_dir, rname)):
        available_repos.append((rname, r["language"]))

print(f"[✓] {len(available_repos)} Repositories Available to Benchmark:")
for idx, (rname, rlang) in enumerate(available_repos, 1):
    print(f"  {idx:02d}. {rname:<20} ({rlang})")

# >>> SET YOUR TARGET REPO HERE <<<
SELECTED_REPO = "fastapi"  # Options: fastapi, tokio, gin, express, gson, requests, etc.
print(f"\n[*] Selected Target Repository: '{SELECTED_REPO}'")

## 2. Core Indexing & Quantization Helper Functions

Here we implement the three chunking and quantization strategies:
- **Quantization**: $1.35$ b/d sign residual quantizer.
- **AST Function Chunking**: Whole-function scope closures.
- **CFG Basic-Block Chunking**: Branching point slicing ($\ge 38$ chars) with DFG def-use chain tags.

In [ ]:
TOKEN_BUDGETS = [500, 1000, 2000, 4000, 8000, 16000, 32000]

def quantize_1bit(x):
    """1-bit sign residual dictionary quantizer: sign(x) / sqrt(d)."""
    return np.sign(x) / np.sqrt(x.shape[-1])

def simple_tokenize(text):
    return [w.lower() for w in re.findall(r'[a-zA-Z0-9_]+', text) if len(w) > 1]

def estimate_tokens(text):
    return max(1, int(len(text) / 3.8))

def bm25_rank(query_tokens, corpus_token_lists, k1=1.5, b=0.75):
    N = len(corpus_token_lists)
    avgdl = np.mean([len(d) for d in corpus_token_lists]) if N > 0 else 1.0
    df = {}
    for doc in corpus_token_lists:
        for t in set(doc):
            df[t] = df.get(t, 0) + 1
    scores = np.zeros(N, dtype=np.float32)
    for t in query_tokens:
        if t not in df: continue
        n_t = df[t]
        idf = math.log(1.0 + (N - n_t + 0.5) / (n_t + 0.5))
        for i, doc in enumerate(corpus_token_lists):
            f = doc.count(t)
            if f > 0:
                denom = f + k1 * (1.0 - b + b * (len(doc) / max(avgdl, 1.0)))
                scores[i] += idf * (f * (k1 + 1.0)) / denom
    return scores

def rrf_fuse(dense_ranks, fts_ranks, k_rrf=60):
    N = len(dense_ranks)
    fused_scores = np.zeros(N, dtype=np.float32)
    for i in range(N):
        r_dense = np.where(dense_ranks == i)[0][0] + 1
        r_fts = np.where(fts_ranks == i)[0][0] + 1
        fused_scores[i] = (1.0 / (k_rrf + r_dense)) + (1.0 / (k_rrf + r_fts))
    return fused_scores

def chunk_file_ast(file_text, rel_path):
    chunks = []
    pattern = r'\n(?=(?:pub(?:\([^)]+\))?\s+)?(?:fn|def|class|struct|trait|func|type|impl|module)\s+)'
    splits = re.split(pattern, file_text)
    cur_line = 1
    for s in splits:
        s_clean = s.strip()
        if not s_clean: continue
        s_lines = s.count("\n")
        start_line = cur_line
        end_line = cur_line + s_lines
        chunks.append({
            "file": rel_path,
            "text": s_clean,
            "start_line": start_line,
            "end_line": end_line,
            "token_count": estimate_tokens(s_clean)
        })
        cur_line = end_line + 1
    if not chunks:
        chunks.append({"file": rel_path, "text": file_text[:1000], "start_line": 1, "end_line": file_text.count('\n')+1, "token_count": estimate_tokens(file_text[:1000])})
    return chunks

def chunk_file_cfg(file_text, rel_path, min_chars=38):
    chunks = []
    pattern = r'\n(?=\s*(?:if\s+|else\s+|match\s+|switch\s+|for\s+|while\s+|try\s+|catch\s+|except\s+|unsafe\s*\{))'
    blocks = re.split(pattern, file_text)
    cur_line = 1
    for b in blocks:
        b_clean = b.strip()
        b_lines = b.count("\n")
        start_line = cur_line
        end_line = cur_line + b_lines
        if len(b_clean) >= min_chars:
            defs = set(re.findall(r'(?:let\s+(?:mut\s+)?|var\s+|:=|self\.)([a-zA-Z0-9_]+)', b_clean))
            dfg_tag = f" // Defs: {', '.join(list(defs)[:3])}" if defs else ""
            full_text = b_clean + dfg_tag
            chunks.append({
                "file": rel_path,
                "text": full_text,
                "start_line": start_line,
                "end_line": end_line,
                "token_count": estimate_tokens(full_text)
            })
        cur_line = end_line + 1
    if not chunks:
        return chunk_file_ast(file_text, rel_path)
    return chunks

## 3. Run Benchmark on Selected Repository

We now load the ground-truth annotations for `SELECTED_REPO`, index its source files, and run all 3 methods live!

In [ ]:
# Find repo metadata
repo_entry = next((r for r in all_repos if r["name"] == SELECTED_REPO), None)
if not repo_entry:
    raise ValueError(f"Repository {SELECTED_REPO} not found in repos.json")

lang = repo_entry["language"]
sub_root = repo_entry.get("benchmark_root", "")
repo_base = os.path.join(repos_dir, SELECTED_REPO)
repo_scan = os.path.join(repo_base, sub_root) if sub_root else repo_base
if not os.path.exists(repo_scan): repo_scan = repo_base

annot_file = os.path.join(annot_dir, f"{SELECTED_REPO}.json")
with open(annot_file, "r") as f:
    tasks = json.load(f)

print(f"[*] Scanning {SELECTED_REPO} ({lang})...")
source_files = []
for dirpath, _, filenames in os.walk(repo_scan):
    if any(p in dirpath for p in [".git", "target", "node_modules", "__pycache__", "docs", "test", "tests", "vendor"]): continue
    for fn in filenames:
        if fn.endswith((".py", ".js", ".ts", ".rs", ".go", ".java", ".rb", ".cpp", ".cs")):
            source_files.append(os.path.join(dirpath, fn))

print(f"[✓] Found {len(source_files)} source files and {len(tasks)} benchmark queries.")

# Read source files
file_contents = {}
for fp in source_files:
    rel_p = os.path.relpath(fp, repo_base)
    with open(fp, "r", errors="ignore") as f:
        file_contents[rel_p] = f.read()

# Build AST & CFG chunks
ast_chunks = []
cfg_chunks = []
for rel_p, content in file_contents.items():
    ast_chunks.extend(chunk_file_ast(content, rel_p))
    cfg_chunks.extend(chunk_file_cfg(content, rel_p))

ast_texts = [c["text"] for c in ast_chunks]
cfg_texts = [c["text"] for c in cfg_chunks]
ast_tok_list = [simple_tokenize(t) for t in ast_texts]
cfg_tok_list = [simple_tokenize(t) for t in cfg_texts]

# Encode Embeddings
print(f"[*] Encoding {len(ast_chunks)} AST chunks and {len(cfg_chunks)} CFG chunks...")
ast_embs_float = potion_code.encode(ast_texts)
cfg_embs_float = potion_code.encode(cfg_texts)

# 1.35 b/d Quantization
ast_embs_135b = quantize_1bit(ast_embs_float)
cfg_embs_135b = quantize_1bit(cfg_embs_float)

# Query Embeddings
queries = [t["query"] for t in tasks]
q_embs_static = potion_code.encode(queries)

print("[✓] Indexing and vector quantization complete!")

## 4. Live Evaluation across All Queries for the Selected Repository

In [ ]:
results_ndcg = {"semble": [], "ast_135b": [], "cfg_champion": []}
results_tokens = {"semble": [], "ast_135b": [], "cfg_champion": []}
results_budget = {m: {b: [] for b in TOKEN_BUDGETS} for m in results_ndcg}
query_trace = []

for qi, task in enumerate(tasks):
    q_text = task["query"]
    q_tok = simple_tokenize(q_text)
    all_targets = task.get("relevant", []) + task.get("secondary", [])

    def check_hit(chunk):
        c_file = chunk["file"]
        for t in all_targets:
            t_path = t if isinstance(t, str) else t.get("path", "")
            if t_path and (t_path in c_file or c_file in t_path or os.path.basename(t_path) == os.path.basename(c_file)):
                if isinstance(t, dict):
                    t_start = t.get("start_line")
                    t_end = t.get("end_line")
                    if t_start is not None and t_end is not None:
                        if chunk["start_line"] <= t_end and chunk["end_line"] >= t_start:
                            return True
                    else:
                        return True
                else:
                    return True
        return False

    # 1. Semble Baseline (Float32 AST + BM25 RRF)
    s1_dense = np.dot(ast_embs_float, q_embs_static[qi])
    s1_bm25 = bm25_rank(q_tok, ast_tok_list)
    s1_fused = rrf_fuse(np.argsort(-s1_dense), np.argsort(-s1_bm25))
    s1_ranked = np.argsort(-s1_fused)
    sem_pos, sem_tok = None, 0
    for pos in range(min(50, len(s1_ranked))):
        c = ast_chunks[s1_ranked[pos]]
        sem_tok += c["token_count"]
        if check_hit(c):
            sem_pos = pos
            break
        if sem_tok >= 32000: break
    ndcg_sem = (1.0 / math.log2(sem_pos + 2)) if (sem_pos is not None and sem_pos < 10) else 0.0
    tok_sem = min(32000, sem_tok) if sem_pos is not None else 32000
    results_ndcg["semble"].append(ndcg_sem)
    results_tokens["semble"].append(tok_sem)
    for b in TOKEN_BUDGETS: results_budget["semble"][b].append(1.0 if tok_sem <= b else 0.0)

    # 2. VQ-bench AST 1.35 b/d
    s2_dense = np.dot(ast_embs_135b, q_embs_static[qi])
    s2_fused = rrf_fuse(np.argsort(-s2_dense), np.argsort(-s1_bm25))
    s2_ranked = np.argsort(-s2_fused)
    ast_pos, ast_tok = None, 0
    for pos in range(min(50, len(s2_ranked))):
        c = ast_chunks[s2_ranked[pos]]
        ast_tok += c["token_count"]
        if check_hit(c):
            ast_pos = pos
            break
        if ast_tok >= 32000: break
    ndcg_ast = (1.0 / math.log2(ast_pos + 2)) if (ast_pos is not None and ast_pos < 10) else 0.0
    tok_ast = min(32000, ast_tok) if ast_pos is not None else 32000
    results_ndcg["ast_135b"].append(ndcg_ast)
    results_tokens["ast_135b"].append(tok_ast)
    for b in TOKEN_BUDGETS: results_budget["ast_135b"][b].append(1.0 if tok_ast <= b else 0.0)

    # 3. VQ-bench CFG Champion
    s3_dense = np.dot(cfg_embs_135b, q_embs_static[qi])
    s3_bm25 = bm25_rank(q_tok, cfg_tok_list) * 1.5
    s3_fused = rrf_fuse(np.argsort(-s3_dense), np.argsort(-s3_bm25))
    s3_ranked = np.argsort(-s3_fused)
    cfg_pos, cfg_tok = None, 0
    for pos in range(min(50, len(s3_ranked))):
        c = cfg_chunks[s3_ranked[pos]]
        cfg_tok += c["token_count"]
        if check_hit(c):
            cfg_pos = pos
            break
        if cfg_tok >= 32000: break
    ndcg_cfg = (1.0 / math.log2(cfg_pos + 2)) if (cfg_pos is not None and cfg_pos < 10) else 0.0
    tok_cfg = min(32000, cfg_tok) if cfg_pos is not None else 32000
    results_ndcg["cfg_champion"].append(ndcg_cfg)
    results_tokens["cfg_champion"].append(tok_cfg)
    for b in TOKEN_BUDGETS: results_budget["cfg_champion"][b].append(1.0 if tok_cfg <= b else 0.0)

    query_trace.append({
        "query": q_text,
        "sem_chunk": ast_chunks[s1_ranked[0]]["text"],
        "cfg_chunk": cfg_chunks[s3_ranked[0]]["text"],
        "sem_tokens": tok_sem,
        "cfg_tokens": tok_cfg
    })

# Print Repository Scorecard
print("=" * 105)
print(f" BENCHMARK RESULTS FOR: {SELECTED_REPO.upper()} ({lang.capitalize()})")
print("=" * 105)
print(f"{'Evaluation Metric':<30} | {'1. Semble Baseline':>22} | {'2. VQ-bench AST 1.35b':>22} | {'3. VQ-bench CFG Champion':>24}")
print("-" * 105)
mean_ndcg_sem = np.mean(results_ndcg['semble'])
mean_ndcg_ast = np.mean(results_ndcg['ast_135b'])
mean_ndcg_cfg = np.mean(results_ndcg['cfg_champion'])
mean_tok_sem = int(np.mean(results_tokens['semble']))
mean_tok_ast = int(np.mean(results_tokens['ast_135b']))
mean_tok_cfg = int(np.mean(results_tokens['cfg_champion']))

print(f"{'NDCG@10 Score':<30} | {mean_ndcg_sem:22.4f} | {mean_ndcg_ast:22.4f} | {mean_ndcg_cfg:24.4f}")
print(f"{'Expected Context Tokens':<30} | {mean_tok_sem:20,d} t | {mean_tok_ast:20,d} t | {mean_tok_cfg:22,d} t")
print(f"{'Context Savings vs Semble':<30} | {'1.0x (Baseline)':>22} | {'1.0x':>22} | {f'{mean_tok_sem/max(1,mean_tok_cfg):.1f}x fewer tokens!':>24}")
print(f"{'Vector Memory Footprint':<30} | {'Float32 (~$2.88/GB)':>22} | {'1.35 b/d ($0.12/GB)':>22} | {'1.35 b/d ($0.12/GB)':>24}")
print(f"{'RAM Reduction vs Float32':<30} | {'0.0%':>22} | {'95.8% Reduction':>22} | {'95.8% Reduction':>24}")
print("=" * 105)

## 5. Recall at Fixed Token Budgets ($500 \to 32\text{k}$ Tokens)

In [ ]:
print("=" * 85)
print(f"{'System / Token Budget':<30} | " + " | ".join([f"{b:>6}t" for b in TOKEN_BUDGETS]))
print("-" * 85)
for m, mlabel in [("semble", "1. Semble Baseline"), ("ast_135b", "2. VQ-bench AST 1.35b"), ("cfg_champion", "3. VQ-bench CFG Champion")]:
    vals = [f"{np.mean(results_budget[m][b]):>7.3f}" for b in TOKEN_BUDGETS]
    print(f"{mlabel:<30} | " + " | ".join(vals))
print("=" * 85)

## 6. Query Inspector: Side-by-Side Snippet Comparison

Inspect the exact retrieved code chunk between Semble's whole-function retrieval and VQ-bench CFG's atomic branch snippet!

In [ ]:
# Inspect first query
trace = query_trace[0]
print(f"Query: '{trace['query']}'\n")
print(f"=== SEMBLE RETRIEVED CHUNK ({trace['sem_tokens']} tokens) ===")
print(trace["sem_chunk"][:400] + "...")
print(f"\n=== VQ-BENCH CFG RETRIEVED CHUNK ({trace['cfg_tokens']} tokens - {trace['sem_tokens']/max(1,trace['cfg_tokens']):.1f}x fewer tokens!) ===")
print(trace["cfg_chunk"][:400] + "...")